In [33]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel
from openai.types.responses import ResponseTextDeltaEvent
import asyncio
load_dotenv(override=True)

True

In [34]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AQ
DeepSeek API Key exists and begins sk-
Groq API Key exists and begins gsk_


## Step 1: Agent workflow

In [35]:
instructions1 = "You are a sales agent working for a General Insurance Company in India, \
a company that provides various types of Insurances like Health, Motor, Travel, Property, Casualty etc\
for retail (individuals) and for the corporates (companies), powered by AI. \
You write professional, serious cold emails to gauge interest and also provide \
explanation on how that product is beneficial to the customer. Overall Email does not exceed 400 words."

instructions2 = "You are a humorous, engaging sales agent working for a General Insurance Company in India, \
a company that provides various types of Insurances like Health, Motor, Property, Casualty etc\
for retail (individuals) and for the corporates (companies), powered by AI.\
You write witty, engaging cold emails that are likely to get a response. \
You give explicit illustrations to the customer on the claim situations of previous customers \
and how they safeguarded their interests due to insurance. Overall Email does not exceed 350 words."

instructions3 = "You are a busy sales agent working for a General Insurance Company in India, \
a company that provides various types of Insurances like Health, Motor, Property, Casualty etc\
for retail (individuals) and for the corporates (companies), powered by AI. \
You write concise, to the point cold emails.  Overall Email does not exceed 100 words."

In [36]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
#DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [38]:

#deepseek_client = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

#deepseek_model = OpenAIChatCompletionsModel(model="deepseek-flash", openai_client=deepseek_client)
gemini_model = OpenAIChatCompletionsModel(model="gemini-flash-lite-latest", openai_client=gemini_client)
llama3_3_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)

In [39]:
sales_agent1 = Agent(name="DeepSeek Sales Agent", instructions=instructions1, model=gemini_model)
sales_agent2 =  Agent(name="Gemini Sales Agent", instructions=instructions2, model=gemini_model)
sales_agent3  = Agent(name="Llama3.3 Sales Agent",instructions=instructions3,model=gemini_model)

In [40]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

### Seeing what Email the LLMs are writing

In [30]:
## Indenting this for now, can be used to check if the LLMs are generating the response or not.

#result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
#async for event in result.stream_events():
#    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
#        print(event.data.delta, end="", flush=True)
#        

### If we have to call all the 3 Agents Parallely (Skip if using Mgr Agent, as Manager Agent is Doing this work later)

In [31]:
### Indenting this for now, as this is to show parallel WF orchestration w/o any framework

#message = "Write a cold sales email"

#with trace("Parallel cold emails"):
#    results = await asyncio.gather(
#        Runner.run(sales_agent1, message),
#        Runner.run(sales_agent2, message),
#        Runner.run(sales_agent3, message),
#    )

#outputs = [result.final_output for result in results]

#for output in outputs:
#    print(output + "\n\n")


## Before we start - some setup:


Please visit Sendgrid at: https://sendgrid.com/ 

(Sendgrid is a Twilio company for sending emails.)

Please set up an account - it's free! (at least, for me, right now).Once you've created an account, click on: Settings (left sidebar) >> API Keys >> Create API Key (button on top right)

Copy the key to the clipboard, then add a new line to your .env file:
`SENDGRID_API_KEY=xxxx`

And also, within SendGrid, go to:
Settings (left sidebar) >> Sender Authentication >> "Verify a Single Sender"  
and verify that your own email address is a real email address, so that SendGrid can send emails for you.


In [ ]:
# Let's just check emails are working for you

#def send_test_email():
#    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
#    from_email = Email("srijita_agarwal@yahoo.co.in")  # Change to your verified sender
#    to_email = To("srijita.dsouza@niveussolutions.com")  # Change to your recipient
#    content = Content("text/plain", "This is an important test email")
#    mail = Mail(from_email, to_email, "Test email", content).get()
#    response = sg.client.mail.send.post(request_body=mail)
#    print(content)
#    print(response.status_code)

#send_test_email()



### Making a Function into a Tool by adding @funtion_tool 

In [41]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("srijita_agarwal@yahoo.co.in")  # Change to your verified sender
    to_email = To("srijita.dsouza@niveussolutions.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

### Email Subject Writer, HTML converter - Tools and Agents

Send Email - Function as a Tool
Subject Writer and Email Converter - Agent as a Tool

In [42]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model=gemini_model)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model=gemini_model)
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [43]:
email_tools = [subject_tool, html_tool, send_html_email]

In [44]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=email_tools,
    model=llama3_3_model,
    handoff_description="Convert an email to HTML and send it")

In [45]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

In [ ]:
sales_manager_instructions = """
You are a Sales Manager at a General Insurance Company in India. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


### Showtime [Excluded for now as adding the Guardrails]

In [ ]:
#sales_manager = Agent(
#    name="Sales Manager",
#    instructions=sales_manager_instructions,
#    tools=tools,
#    handoffs=handoffs,
#    model=gemini_model)

#message = "Send out a cold sales email addressed to Dear CEO from Alice"

#with trace("Automated SDR"):
#    result = await Runner.run(sales_manager, message)

## Check out the trace:

https://platform.openai.com/traces

In [48]:
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

guardrail_agent = Agent( 
    name="Name check",
    instructions="Check if the user is including someone's personal name in what they want you to do.",
    output_type=NameCheckOutput,
    model=gemini_model
)

In [49]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)

### Run below to see the Guardrail Tipping [Excluded for now]

In [ ]:
careful_sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model=gemini_model,
    input_guardrails=[guardrail_against_name]
    )

#message = "Send out a cold sales email addressed to Dear CEO Abhijit from Alice"

#with trace("Protected(Name) Automated SDR"):
#    result = await Runner.run(careful_sales_manager, message)

### All Agents + Tools Coming Together to Achieve the task

In [ ]:

message = "Send out a cold sales email addressed to Dear CEO from Head of Business Development"

with trace("Protected(Without name) Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)